# Day 4 – Sentence Transformers, Embeddings & Vector Search

**Topics:** Sentence Transformers (free) · Text Embeddings · Cosine Similarity · FAISS · Vector DB Comparison (FAISS vs ChromaDB vs Pinecone)

| Section | Script | What it covers |
|---|---|---|
| 1 | `01_embeddings_basics` | Load `all-MiniLM-L6-v2`, encode sentences, shapes, similarity matrix |
| 2 | `02_cosine_similarity` | Manual cosine sim, sklearn matrix, `util.cos_sim`, duplicate detection |
| 3 | `03_faiss_search` | FlatL2, FlatIP, IVFFlat, HNSWFlat, save/load, benchmark |
| 4 | `04_metadata_filtering` | Post-filter & pre-filter patterns, AND/OR logic, persist JSON metadata |
| 5 | `05_semantic_engine` | Full `SemanticSearchEngine` class with incremental adds and save/load |
| 6 | ChromaDB demo | Built-in metadata filter comparison |


## 0. Install Dependencies


In [ ]:
# Run once to install all required packages
# (already available in Google Colab; run locally if needed)
!pip install sentence-transformers faiss-cpu scikit-learn -q


---
## 1. Embeddings Basics with Sentence Transformers

Key ideas:
- Load a pre-trained `SentenceTransformer` model (`all-MiniLM-L6-v2` — 90 MB, 384-dim)
- `model.encode(sentences)` → **numpy array** of shape `(n, 384)`
- `normalize_embeddings=True` → unit-length vectors → dot product == cosine similarity
- Build a **pairwise similarity matrix** using matrix multiplication

**Architecture:**
```
Text → Tokenizer → Transformer → Token Embeddings (seq_len × 768)
                                        ↓ Mean Pooling
                              Sentence Embedding (384,)
```


In [ ]:
"""
Day 4 – Script 01: Embeddings Basics with Sentence Transformers
================================================================
This script demonstrates:
  - Loading a Sentence Transformer model
  - Generating sentence embeddings
  - Inspecting embedding shapes and values
  - Comparing embeddings visually (dot-product heatmap without matplotlib)
  - Showing how similar sentences cluster together

Run:
    python 01_embeddings_basics.py

Requirements:
    pip install sentence-transformers
"""

import numpy as np
from sentence_transformers import SentenceTransformer

# ─────────────────────────────────────────────────────────────────────────────
# 1. Load the model
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("Day 4 – Embeddings Basics with Sentence Transformers")
print("=" * 60)

MODEL_NAME = "all-MiniLM-L6-v2"   # 90MB, 384-dim, fast & free
print(f"\nLoading model: {MODEL_NAME} (downloads on first run)...")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded ✓")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Encode a few sentences
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 1. Single Sentence Embedding ──")

sentence = "Artificial intelligence is transforming the world."
embedding = model.encode(sentence)

print(f"Sentence    : {sentence}")
print(f"Shape       : {embedding.shape}")          # (384,)
print(f"Dtype       : {embedding.dtype}")
print(f"Min / Max   : {embedding.min():.4f} / {embedding.max():.4f}")
print(f"Norm (L2)   : {np.linalg.norm(embedding):.4f}")   # ~1 if normalised
print(f"First 8 dims: {embedding[:8].round(4)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Batch encoding
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 2. Batch Encoding ──")

sentences = [
    "The cat sat on the mat.",
    "A feline rested on a rug.",       # semantically similar to #0
    "Dogs love to play fetch.",
    "The stock market crashed today.",
    "Machine learning uses data to make predictions.",
    "Neural networks are inspired by the human brain.",
]

embeddings = model.encode(sentences, show_progress_bar=False)
print(f"Input sentences : {len(sentences)}")
print(f"Embeddings shape: {embeddings.shape}")   # (6, 384)

# ─────────────────────────────────────────────────────────────────────────────
# 4. L2-normalised embeddings (unit vectors)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 3. L2-Normalised Embeddings ──")

norm_embs = model.encode(sentences, normalize_embeddings=True)
norms = np.linalg.norm(norm_embs, axis=1)
print(f"All norms after normalisation: {norms.round(4)}")
print(f"(Should all be ~1.0000)")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Raw dot-product similarity matrix (since vectors are L2-normalised,
#    dot product == cosine similarity)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 4. Pairwise Similarity (dot-product on normalised vectors) ──")

sim_matrix = norm_embs @ norm_embs.T   # (6, 6)
print("\nSimilarity matrix (rows/cols = sentences 0-5):\n")

# Print nicely aligned
header = "        " + "  ".join(f"  s{i}" for i in range(len(sentences)))
print(header)
for i, row in enumerate(sim_matrix):
    row_str = "  ".join(f"{v:5.3f}" for v in row)
    print(f"  s{i}  [ {row_str} ]")

print("\nKey observations:")
print(f"  s0 ↔ s1 (cat/feline)   : {sim_matrix[0,1]:.4f}  ← should be HIGH")
print(f"  s4 ↔ s5 (ML/neural)    : {sim_matrix[4,5]:.4f}  ← should be HIGH")
print(f"  s0 ↔ s3 (cat/stock mkt): {sim_matrix[0,3]:.4f}  ← should be LOW")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Comparing different model sizes
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 5. Model Comparison ──")

model_info = [
    ("all-MiniLM-L6-v2",     "384-dim, fast, 90MB"),
    ("all-mpnet-base-v2",    "768-dim, high quality, 420MB"),
]
print(f"{'Model':<35} {'Dimension':<12} {'Note'}")
print("-" * 70)
for name, note in model_info:
    try:
        m = SentenceTransformer(name)
        dim = m.encode("test").shape[0]
        print(f"{name:<35} {dim:<12} {note}")
    except Exception as e:
        print(f"{name:<35} {'N/A':<12} (load error: {e})")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Encode a large batch efficiently
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 6. Large Batch Encoding ──")

# Simulate a larger corpus
large_corpus = [f"This is sentence number {i} about topic {i % 5}." for i in range(100)]
model_fast = SentenceTransformer("all-MiniLM-L6-v2")

import time
start = time.time()
large_embs = model_fast.encode(
    large_corpus,
    batch_size=32,
    show_progress_bar=False,
    normalize_embeddings=True,
)
elapsed = time.time() - start

print(f"Encoded {len(large_corpus)} sentences in {elapsed:.2f}s")
print(f"Throughput: {len(large_corpus)/elapsed:.0f} sentences/sec")
print(f"Output shape: {large_embs.shape}")

print("\n✅  Embeddings basics demo complete.")

---
## 2. Cosine Similarity – Deep Dive

**Formula:**
```
cosine_similarity(A, B) = (A · B) / (||A|| × ||B||)
```
- Range: **−1** (opposite) → **0** (unrelated) → **+1** (identical direction)
- Scores above **0.7** are typically considered semantically similar

This section covers:
- Manual numpy implementation
- sklearn `cosine_similarity` for full pairwise matrix
- `sentence_transformers.util.cos_sim` (PyTorch tensor API)
- Semantic duplicate / near-duplicate detection
- Score interpretation guide


In [ ]:
"""
Day 4 – Script 02: Cosine Similarity Deep Dive
===============================================
This script demonstrates:
  1. Manual cosine similarity computation (numpy)
  2. Pairwise similarity matrix with sklearn
  3. sentence_transformers util.cos_sim (PyTorch tensors)
  4. Finding top-k most similar sentences to a query
  5. Semantic Textual Similarity (STS) scoring
  6. The effect of L2 normalisation on dot product

Run:
    python 02_cosine_similarity.py

Requirements:
    pip install sentence-transformers scikit-learn
"""

import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 60)
print("Day 4 – Script 02: Cosine Similarity")
print("=" * 60)

model = SentenceTransformer("all-MiniLM-L6-v2")

# ─────────────────────────────────────────────────────────────────────────────
# 1. Manual cosine similarity using numpy
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 1. Manual Cosine Similarity (numpy) ──")

def cosine_sim_numpy(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two 1-D vectors."""
    dot   = np.dot(a, b)
    norm  = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / norm) if norm > 0 else 0.0

pairs = [
    ("I love playing football.",   "Soccer is my favourite sport."),    # similar
    ("The sun is a star.",         "Stars are massive balls of gas."),  # related
    ("I love playing football.",   "The recipe requires two eggs."),    # unrelated
]

for s1, s2 in pairs:
    e1, e2 = model.encode([s1, s2])
    score  = cosine_sim_numpy(e1, e2)
    print(f"  {score:+.4f}  |  '{s1[:35]}…' ↔ '{s2[:35]}…'")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Pairwise similarity matrix (sklearn)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 2. Pairwise Similarity Matrix (sklearn) ──")

sentences = [
    "Deep learning is a type of machine learning.",
    "Machine learning enables computers to learn from data.",
    "The Eiffel Tower stands 330 metres tall.",
    "Neural networks are the foundation of deep learning.",
    "Paris is known for its iconic tower.",
]

embs   = model.encode(sentences)
matrix = cosine_similarity(embs)   # shape (5, 5)

labels = [f"s{i}" for i in range(len(sentences))]
print(f"\n{'':6s}" + "  ".join(f"{l:>6}" for l in labels))
for i, row in enumerate(matrix):
    row_str = "  ".join(f"{v:6.3f}" for v in row)
    print(f"  {labels[i]}  {row_str}")

print("\nSentences (for reference):")
for i, s in enumerate(sentences):
    print(f"  s{i}: {s}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. sentence_transformers util.cos_sim (PyTorch tensor API)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 3. util.cos_sim (sentence_transformers PyTorch API) ──")

query      = "What is deep learning?"
candidates = [
    "Deep learning is a subset of machine learning using neural networks.",
    "The Great Wall of China stretches thousands of miles.",
    "Neural networks learn hierarchical representations.",
    "I enjoy hiking in the mountains.",
    "Backpropagation is the algorithm used to train neural networks.",
]

query_emb     = model.encode(query,      convert_to_tensor=True)
candidate_embs = model.encode(candidates, convert_to_tensor=True)

scores = util.cos_sim(query_emb, candidate_embs)   # (1, 5) tensor

print(f"\nQuery: '{query}'\n")
print(f"{'Rank':<5} {'Score':>7}  Candidate")
print("-" * 70)

# Sort by score descending
ranked = sorted(enumerate(scores[0].tolist()), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  {rank:<3}  {score:+.4f}  {candidates[idx]}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Top-k most similar pairs in a corpus (semantic duplicate detection)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 4. Finding Semantic Duplicates / Near-Duplicates ──")

corpus = [
    "How do I install Python?",
    "What is the best way to set up Python?",
    "The moon orbits the Earth.",
    "Earth is orbited by the Moon.",
    "How do neural networks work?",
    "Can you explain how neural nets function?",
    "What is the capital of France?",
    "Where is Paris located?",
]

corpus_embs = model.encode(corpus, normalize_embeddings=True)
sim         = cosine_similarity(corpus_embs)

THRESHOLD = 0.75
print(f"\nPairs with cosine similarity ≥ {THRESHOLD}:\n")
for i in range(len(corpus)):
    for j in range(i + 1, len(corpus)):
        score = sim[i, j]
        if score >= THRESHOLD:
            print(f"  [{score:.4f}]")
            print(f"    A: {corpus[i]}")
            print(f"    B: {corpus[j]}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. L2 normalisation: dot product == cosine similarity
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 5. Dot Product vs Cosine Sim (after L2 normalisation) ──")

s1, s2 = "I love cats.", "Cats are my favourite animals."
e1_raw = model.encode([s1, s2])
e1_norm = model.encode([s1, s2], normalize_embeddings=True)

raw_dot    = float(e1_raw[0] @ e1_raw[1])
norm_dot   = float(e1_norm[0] @ e1_norm[1])
cos_manual = cosine_sim_numpy(e1_raw[0], e1_raw[1])

print(f"  Raw dot product               : {raw_dot:.6f}")
print(f"  Cosine similarity (manual)    : {cos_manual:.6f}")
print(f"  Dot product after L2-norm     : {norm_dot:.6f}")
print(f"  All three match (for unit vec): {abs(cos_manual - norm_dot) < 1e-5}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Similarity score interpretation guide
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 6. Score Interpretation Guide ──")

test_pairs = [
    ("I love cats.", "I love cats."),                                  # identical
    ("I love cats.", "I adore felines."),                              # paraphrase
    ("Python is a programming language.", "Python can be venomous."), # polysemy
    ("I love cats.", "The economy grew by 3% last quarter."),          # unrelated
]

print(f"\n{'Score':>7}  {'Label':<15}  Pair")
print("-" * 75)
for s1, s2 in test_pairs:
    e1, e2 = model.encode([s1, s2], normalize_embeddings=True)
    score  = float(e1 @ e2)
    if score >= 0.9:
        label = "near-identical"
    elif score >= 0.7:
        label = "very similar"
    elif score >= 0.5:
        label = "somewhat similar"
    elif score >= 0.3:
        label = "weakly related"
    else:
        label = "unrelated"
    print(f"  {score:+.4f}  {label:<15}  '{s1[:25]}' ↔ '{s2[:25]}'")

print("\n✅  Cosine similarity demo complete.")

---
## 3. FAISS Similarity Search

**FAISS** (Facebook AI Similarity Search) enables lightning-fast vector search.

| Index | Type | When to Use |
|---|---|---|
| `IndexFlatL2` | Exact, Euclidean distance | Tiny datasets, max accuracy |
| `IndexFlatIP` | Exact, Inner Product (cosine) | Normalised vectors, small-medium |
| `IndexIVFFlat` | Approximate (Voronoi clusters) | 10 k+ documents |
| `IndexHNSWFlat` | Approximate (graph-based) | Fast queries, higher memory |

**Key parameters:**
- `nlist` — number of IVF clusters (more = better partitioning, slower training)
- `nprobe` — clusters to search at query time (higher = more accurate, slower)
- `M` — HNSW graph connections per node (higher = more accurate, more memory)


In [ ]:
"""
Day 4 – Script 03: FAISS Similarity Search
===========================================
This script demonstrates:
  1. IndexFlatL2  – exact L2 (Euclidean) distance search
  2. IndexFlatIP  – exact Inner Product (cosine sim after L2-norm)
  3. IndexIVFFlat – approximate search (faster for large corpora)
  4. IndexHNSWFlat – graph-based ANN search
  5. Save and reload an index to / from disk
  6. Benchmarking: exact vs approximate at scale

Run:
    python 03_faiss_search.py

Requirements:
    pip install sentence-transformers faiss-cpu
"""

import numpy as np
import faiss
import time
from sentence_transformers import SentenceTransformer

print("=" * 60)
print("Day 4 – Script 03: FAISS Similarity Search")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Corpus and embeddings
# ─────────────────────────────────────────────────────────────────────────────
model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    # Tech / AI
    "FAISS enables fast similarity search over millions of vectors.",
    "Sentence Transformers produce dense semantic embeddings.",
    "PyTorch is a popular deep learning framework developed by Meta.",
    "TensorFlow is Google's open-source machine learning platform.",
    "Cosine similarity measures the angle between two embedding vectors.",
    "Transformers revolutionised NLP with the attention mechanism.",
    "BERT is a powerful bidirectional encoder from Google.",
    "GPT models generate text by predicting the next token.",
    "Vector databases store and retrieve high-dimensional embeddings.",
    "RAG combines retrieval with large language model generation.",
    # Travel
    "Paris is the capital of France and home to the Eiffel Tower.",
    "Rome has ancient ruins including the Colosseum and the Forum.",
    "Tokyo is a bustling city blending tradition and modernity.",
    "New York City is famous for Central Park and Times Square.",
    "Barcelona is known for Gaudí's unique architecture.",
    # Science
    "Quantum computing uses qubits to process information.",
    "Black holes are regions where gravity is so strong light cannot escape.",
    "DNA carries genetic information using four nucleotide bases.",
    "Photosynthesis converts sunlight into chemical energy in plants.",
    "The speed of light in a vacuum is approximately 3×10⁸ m/s.",
]

print(f"\nEncoding {len(documents)} documents...")
embeddings = model.encode(documents, normalize_embeddings=True, show_progress_bar=False)
embeddings = embeddings.astype("float32")
d          = embeddings.shape[1]
print(f"Embedding shape: {embeddings.shape}  (n_docs × dim)")

QUERY = "How do vector search engines work?"
q_emb = model.encode([QUERY], normalize_embeddings=True).astype("float32")

K = 5  # top-k results

def print_results(label: str, distances, indices, docs):
    print(f"\n── {label} ──")
    print(f"Query: '{QUERY}'\n")
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        if idx == -1:
            continue
        print(f"  {rank}. [{dist:.4f}] {docs[idx]}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. IndexFlatL2 – Exact L2 distance (smaller = more similar)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 1. IndexFlatL2 (Exact Euclidean Distance) ──")

index_l2 = faiss.IndexFlatL2(d)
index_l2.add(embeddings)

distances_l2, indices_l2 = index_l2.search(q_emb, K)
print_results("IndexFlatL2 results (lower distance = more similar)", distances_l2, indices_l2, documents)

# ─────────────────────────────────────────────────────────────────────────────
# 2. IndexFlatIP – Exact Inner Product (cosine sim for L2-normalised vectors)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 2. IndexFlatIP (Exact Cosine Similarity) ──")

index_ip = faiss.IndexFlatIP(d)
index_ip.add(embeddings)

distances_ip, indices_ip = index_ip.search(q_emb, K)
print_results("IndexFlatIP results (higher score = more similar)", distances_ip, indices_ip, documents)

# ─────────────────────────────────────────────────────────────────────────────
# 3. IndexIVFFlat – Approximate, faster for large corpora
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 3. IndexIVFFlat (Approximate, Inverted File Index) ──")

nlist     = 5       # Number of Voronoi cells (clusters)
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# IVF index must be trained before adding vectors
print("Training IVF index...")
index_ivf.train(embeddings)
index_ivf.add(embeddings)

index_ivf.nprobe = 3   # Number of clusters to search (higher = more accurate, slower)

distances_ivf, indices_ivf = index_ivf.search(q_emb, K)
print_results(f"IndexIVFFlat (nlist={nlist}, nprobe={index_ivf.nprobe})", distances_ivf, indices_ivf, documents)

print(f"\n  IVF nprobe={index_ivf.nprobe}: searches {index_ivf.nprobe}/{nlist} clusters")
print("  → Increase nprobe for higher accuracy, decrease for speed")

# ─────────────────────────────────────────────────────────────────────────────
# 4. IndexHNSWFlat – Graph-based ANN (very fast, high accuracy)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 4. IndexHNSWFlat (Hierarchical Navigable Small World) ──")

M         = 16    # Number of connections per node in the graph (higher = more accurate)
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.add(embeddings)

distances_hnsw, indices_hnsw = index_hnsw.search(q_emb, K)
print_results(f"IndexHNSWFlat (M={M})", distances_hnsw, indices_hnsw, documents)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Save and reload a FAISS index
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 5. Save & Reload FAISS Index ──")

faiss.write_index(index_ip, "demo_index.faiss")
print("Index saved to demo_index.faiss")

loaded_index = faiss.read_index("demo_index.faiss")
print(f"Index reloaded: {loaded_index.ntotal} vectors, dim={loaded_index.d}")

# Verify search still works
d_r, i_r = loaded_index.search(q_emb, 3)
print(f"Verification search top-1: [{d_r[0][0]:.4f}] {documents[i_r[0][0]]}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Benchmark: Exact (FlatIP) vs Approximate (IVF) at scale
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 6. Benchmark: Exact vs Approximate Search ──")

# Simulate a larger corpus (1000 synthetic vectors)
rng        = np.random.default_rng(42)
big_vecs   = rng.standard_normal((1_000, d)).astype("float32")
# L2-normalise
norms_big  = np.linalg.norm(big_vecs, axis=1, keepdims=True)
big_vecs  /= norms_big

big_query  = big_vecs[:1]   # Use first vector as query
K_bench    = 10

# Exact
idx_exact = faiss.IndexFlatIP(d)
idx_exact.add(big_vecs)
t0 = time.perf_counter()
for _ in range(20):
    idx_exact.search(big_query, K_bench)
exact_ms = (time.perf_counter() - t0) / 20 * 1000

# IVF Approximate
nlist_bench = 50
q_bench = faiss.IndexFlatIP(d)
idx_approx = faiss.IndexIVFFlat(q_bench, d, nlist_bench, faiss.METRIC_INNER_PRODUCT)
idx_approx.train(big_vecs)
idx_approx.add(big_vecs)
idx_approx.nprobe = 10

t0 = time.perf_counter()
for _ in range(20):
    idx_approx.search(big_query, K_bench)
approx_ms = (time.perf_counter() - t0) / 20 * 1000

print(f"\n  Dataset size  : 1,000 vectors × {d} dims")
print(f"  IndexFlatIP   : {exact_ms:.3f} ms/query  (exact)")
print(f"  IndexIVFFlat  : {approx_ms:.3f} ms/query  (approx, nprobe={idx_approx.nprobe})")
print(f"  Speedup       : {exact_ms/approx_ms:.1f}x  (grows with dataset size)")

print("""
FAISS Index Cheat Sheet:
  IndexFlatL2   → Exact, L2 distance     → Best accuracy, slowest
  IndexFlatIP   → Exact, cosine sim      → Best accuracy for normalised vectors
  IndexIVFFlat  → Approx (IVF)           → Good balance accuracy/speed for 10k+ docs
  IndexHNSWFlat → Approx (graph-based)   → Very fast queries, higher memory
""")

print("✅  FAISS similarity search demo complete.")

---
## 4. Metadata Filtering with FAISS

FAISS stores **only float32 vectors** — no text, no metadata.
The document index in `corpus[]` always equals the FAISS internal ID.

**Pattern 1 – Post-filter (most common):**
```
1. Over-fetch k*6 candidates from FAISS
2. Iterate and discard docs that fail metadata checks
3. Return first k passing docs
```

**Pattern 2 – Pre-filter (better for large corpora):**
```
1. Build a separate FAISS index per category
2. Query only the relevant sub-index
```

> For built-in metadata filtering → use **ChromaDB** or **Weaviate** instead.


In [ ]:
"""
Day 4 – Script 04: Metadata Filtering with FAISS
=================================================
FAISS stores only vectors — no metadata.
This script demonstrates the standard pattern for adding metadata:

  1. Maintain a parallel Python list (metadata store)
  2. Post-filter results by metadata field
  3. Pre-filter pattern (restrict candidate set before FAISS search)
  4. Combining multiple filter conditions (AND / OR logic)
  5. Persisting metadata alongside the FAISS index (JSON)

Run:
    python 04_metadata_filtering.py

Requirements:
    pip install sentence-transformers faiss-cpu
"""

import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("=" * 60)
print("Day 4 – Script 04: Metadata Filtering with FAISS")
print("=" * 60)

model = SentenceTransformer("all-MiniLM-L6-v2")

# ─────────────────────────────────────────────────────────────────────────────
# Our "database" — a list of dicts with text + metadata
# ─────────────────────────────────────────────────────────────────────────────
corpus = [
    # --- AI / ML ---
    {"text": "FAISS allows fast nearest-neighbour search at scale.",         "category": "ai",       "level": "advanced", "year": 2023, "lang": "en"},
    {"text": "Sentence Transformers create dense sentence embeddings.",       "category": "ai",       "level": "beginner", "year": 2022, "lang": "en"},
    {"text": "PyTorch is widely used for training neural networks.",          "category": "ai",       "level": "intermediate", "year": 2022, "lang": "en"},
    {"text": "Cosine similarity finds semantically related documents.",       "category": "ai",       "level": "beginner", "year": 2023, "lang": "en"},
    {"text": "Large language models are trained on massive text corpora.",     "category": "ai",       "level": "advanced", "year": 2023, "lang": "en"},
    {"text": "ChromaDB is an open-source vector database for embeddings.",    "category": "ai",       "level": "intermediate", "year": 2023, "lang": "en"},
    # --- Travel ---
    {"text": "The Eiffel Tower is a symbol of Paris and French culture.",     "category": "travel",   "level": "beginner", "year": 2021, "lang": "en"},
    {"text": "Kyoto is famous for its temples and traditional geisha culture.","category": "travel",  "level": "beginner", "year": 2021, "lang": "en"},
    {"text": "Santorini is a volcanic island known for its white buildings.",  "category": "travel",   "level": "beginner", "year": 2022, "lang": "en"},
    {"text": "The Amazon rainforest spans nine South American countries.",     "category": "travel",   "level": "intermediate", "year": 2022, "lang": "en"},
    # --- Science ---
    {"text": "Quantum entanglement allows particles to correlate over distances.", "category": "science", "level": "advanced", "year": 2023, "lang": "en"},
    {"text": "DNA replication ensures genetic information is passed to daughter cells.", "category": "science", "level": "intermediate", "year": 2022, "lang": "en"},
    {"text": "Black holes form when massive stars collapse under their gravity.", "category": "science", "level": "beginner", "year": 2021, "lang": "en"},
    # --- Programming ---
    {"text": "Python decorators are a powerful metaprogramming tool.",        "category": "programming", "level": "intermediate", "year": 2022, "lang": "en"},
    {"text": "JavaScript async/await simplifies asynchronous code.",          "category": "programming", "level": "beginner",      "year": 2023, "lang": "en"},
    {"text": "Rust guarantees memory safety without garbage collection.",      "category": "programming", "level": "advanced",      "year": 2023, "lang": "en"},
]

# ─────────────────────────────────────────────────────────────────────────────
# Build the FAISS index (pure vector storage)
# ─────────────────────────────────────────────────────────────────────────────
texts      = [doc["text"] for doc in corpus]
embeddings = model.encode(texts, normalize_embeddings=True).astype("float32")
d          = embeddings.shape[1]

index = faiss.IndexFlatIP(d)
index.add(embeddings)
print(f"\nIndex ready: {index.ntotal} vectors of dim {d}")
print(f"Metadata store: {len(corpus)} entries")

# ─────────────────────────────────────────────────────────────────────────────
# 1. Post-filter search function
# ─────────────────────────────────────────────────────────────────────────────
def search(
    query: str,
    k: int = 5,
    filters: dict = None,    # {"category": "ai"} or {"level": "beginner", "year": 2023}
    logic: str = "AND",      # "AND" or "OR"
) -> list[dict]:
    """
    Embed query, retrieve top candidates from FAISS,
    then post-filter by metadata.
    """
    query_emb  = model.encode([query], normalize_embeddings=True).astype("float32")
    candidates = min(index.ntotal, k * 6)   # over-fetch to allow for filtering
    distances, indices = index.search(query_emb, candidates)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < 0:
            continue
        doc = corpus[idx]
        if filters:
            matches = [doc.get(fk) == fv for fk, fv in filters.items()]
            if logic == "AND" and not all(matches):
                continue
            if logic == "OR" and not any(matches):
                continue
        results.append({"score": round(float(dist), 4), **doc})
        if len(results) >= k:
            break
    return results

def print_results(title: str, results: list):
    print(f"\n── {title} ──")
    if not results:
        print("  (no results after filtering)")
        return
    for r in results:
        meta = f"[{r['category']} | {r['level']} | {r['year']}]"
        print(f"  [{r['score']:.4f}] {meta:35s} {r['text'][:60]}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Demo: no filter vs filtered
# ─────────────────────────────────────────────────────────────────────────────
QUERY = "fast search over high-dimensional vectors"

print_results(
    "No filter",
    search(QUERY, k=4),
)

print_results(
    "Filter: category=ai",
    search(QUERY, k=4, filters={"category": "ai"}),
)

print_results(
    "Filter: category=ai AND level=beginner",
    search(QUERY, k=4, filters={"category": "ai", "level": "beginner"}),
)

print_results(
    "Filter: category=ai AND year=2023",
    search(QUERY, k=4, filters={"category": "ai", "year": 2023}),
)

# ─────────────────────────────────────────────────────────────────────────────
# 3. OR logic
# ─────────────────────────────────────────────────────────────────────────────
print_results(
    "Filter: category=travel OR category=science  (OR logic)",
    search("interesting natural phenomena", k=4,
           filters={"category": "travel", "level": "advanced"}, logic="OR"),
)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Pre-filter pattern (build a sub-index for a category)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Pre-filter Pattern: Build Sub-Index per Category ──")

def build_subindex(category: str):
    """Return a FAISS index and local metadata list for one category."""
    subset     = [(i, doc) for i, doc in enumerate(corpus) if doc["category"] == category]
    local_meta = [doc for _, doc in subset]
    local_embs = np.stack([embeddings[i] for i, _ in subset]).astype("float32")
    sub_idx    = faiss.IndexFlatIP(d)
    sub_idx.add(local_embs)
    return sub_idx, local_meta

ai_index, ai_meta = build_subindex("ai")
print(f"AI sub-index contains {ai_index.ntotal} vectors")

q_emb   = model.encode([QUERY], normalize_embeddings=True).astype("float32")
dists, idxs = ai_index.search(q_emb, 3)

print(f"\nTop-3 AI results for: '{QUERY}'")
for dist, idx in zip(dists[0], idxs[0]):
    print(f"  [{dist:.4f}] {ai_meta[idx]['text']}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Persist index + metadata to disk
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 5. Persisting Index + Metadata ──")

INDEX_PATH = "corpus_index.faiss"
META_PATH  = "corpus_meta.json"

faiss.write_index(index, INDEX_PATH)
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

print(f"Saved FAISS index → {INDEX_PATH}")
print(f"Saved metadata    → {META_PATH}")

# Reload
loaded_index = faiss.read_index(INDEX_PATH)
with open(META_PATH, encoding="utf-8") as f:
    loaded_meta = json.load(f)

print(f"Reloaded: {loaded_index.ntotal} vectors, {len(loaded_meta)} metadata entries")

# Verify
dists2, idxs2 = loaded_index.search(q_emb, 3)
print("\nVerification search top-1 after reload:")
print(f"  [{dists2[0][0]:.4f}] {loaded_meta[idxs2[0][0]]['text']}")

print("""
Key Takeaways – Metadata Filtering with FAISS:
  ─ FAISS stores only float32 vectors (no text, no metadata)
  ─ Always maintain a parallel Python list of metadata dicts
  ─ Document position in corpus list == FAISS internal ID
  ─ Post-filter: over-fetch (k*4 or k*6), then apply filters
  ─ Pre-filter: build a separate index per category (faster for big corpora)
  ─ For built-in metadata filtering → use ChromaDB or Weaviate instead
""")

print("✅  Metadata filtering demo complete.")

---
## 5. Full Semantic Search Engine

A reusable `SemanticSearchEngine` class that wraps Sentence Transformers + FAISS:

| Method | What it does |
|---|---|
| `build_index(documents)` | Encode all docs and build `IndexFlatIP` |
| `add_documents(docs)` | Incrementally add more docs without rebuilding |
| `search(query, k, filters)` | Semantic search with optional AND metadata filter |
| `save(path)` | Write `.faiss` index + `_meta.json` to disk |
| `load(path)` | Reload both files |
| `stats()` | Count docs, embedding dim, per-category breakdown |

> **Note:** The interactive REPL at the bottom of the script is terminal-only.
> The cells above it will run fine in a notebook.


In [ ]:
"""
Day 4 – Script 05: Full Semantic Search Engine (End-to-End)
===========================================================
A reusable SemanticSearchEngine class that wraps:
  - Sentence Transformers (encoding)
  - FAISS IndexFlatIP (search)
  - Parallel metadata store
  - Save / Load (index + metadata)
  - Metadata filtering (AND logic)
  - Interactive REPL demo

Run:
    python 05_semantic_engine.py

Requirements:
    pip install sentence-transformers faiss-cpu
"""

import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ═══════════════════════════════════════════════════════════════════════════════
# SemanticSearchEngine class
# ═══════════════════════════════════════════════════════════════════════════════

class SemanticSearchEngine:
    """
    Lightweight semantic search engine backed by Sentence Transformers + FAISS.

    Usage:
        engine = SemanticSearchEngine()
        engine.build_index(documents)          # list of dicts with "text" key
        results = engine.search("my query", k=5, filters={"category": "ai"})
        engine.save("my_engine")
        engine.load("my_engine")
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading model: {model_name}...")
        self.model    = SentenceTransformer(model_name)
        self.index    = None
        self.metadata: list[dict] = []
        self.d        = None
        print("Model ready ✓")

    # ─────────────────────────────────────────────────────────────────────────
    def build_index(
        self,
        documents: list[dict],
        text_key: str = "text",
        batch_size: int = 64,
        show_progress: bool = True,
    ) -> None:
        """Encode all documents and build a FAISS IndexFlatIP."""
        if not documents:
            raise ValueError("documents list is empty")

        texts          = [doc[text_key] for doc in documents]
        self.metadata  = list(documents)

        print(f"Encoding {len(texts)} documents...")
        embs = self.model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=show_progress,
        ).astype("float32")

        self.d     = embs.shape[1]
        self.index = faiss.IndexFlatIP(self.d)
        self.index.add(embs)
        print(f"Index built: {self.index.ntotal} vectors, dim={self.d}")

    # ─────────────────────────────────────────────────────────────────────────
    def add_documents(self, documents: list[dict], text_key: str = "text") -> None:
        """Incrementally add more documents to an existing index."""
        if self.index is None:
            raise RuntimeError("Call build_index() first.")
        texts = [doc[text_key] for doc in documents]
        embs  = self.model.encode(
            texts, normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")
        self.index.add(embs)
        self.metadata.extend(documents)
        print(f"Added {len(documents)} docs. Total: {self.index.ntotal}")

    # ─────────────────────────────────────────────────────────────────────────
    def search(
        self,
        query: str,
        k: int = 5,
        filters: dict | None = None,
        over_fetch_factor: int = 6,
    ) -> list[dict]:
        """
        Semantic search with optional metadata filtering.

        Args:
            query              : Natural language query string
            k                  : Number of results to return
            filters            : Dict of {field: value} to apply (AND logic)
            over_fetch_factor  : Multiply k by this to allow for filtering

        Returns:
            List of result dicts with 'score' prepended to the metadata.
        """
        if self.index is None or self.index.ntotal == 0:
            return []

        query_emb  = self.model.encode(
            [query], normalize_embeddings=True
        ).astype("float32")

        candidates = min(self.index.ntotal, k * over_fetch_factor)
        distances, indices = self.index.search(query_emb, candidates)

        results: list[dict] = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx < 0:
                continue
            doc = self.metadata[idx]
            if filters:
                if not all(doc.get(fk) == fv for fk, fv in filters.items()):
                    continue
            results.append({"score": round(float(dist), 4), **doc})
            if len(results) >= k:
                break
        return results

    # ─────────────────────────────────────────────────────────────────────────
    def save(self, path_prefix: str) -> None:
        """Save FAISS index (.faiss) and metadata (.json) to disk."""
        faiss.write_index(self.index, f"{path_prefix}.faiss")
        meta_path = f"{path_prefix}_meta.json"
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump({"d": self.d, "documents": self.metadata}, f, indent=2, ensure_ascii=False)
        print(f"Saved → {path_prefix}.faiss  +  {meta_path}")

    # ─────────────────────────────────────────────────────────────────────────
    def load(self, path_prefix: str) -> None:
        """Load FAISS index and metadata from disk."""
        self.index = faiss.read_index(f"{path_prefix}.faiss")
        meta_path  = f"{path_prefix}_meta.json"
        with open(meta_path, encoding="utf-8") as f:
            payload = json.load(f)
        self.d        = payload["d"]
        self.metadata = payload["documents"]
        print(f"Loaded ← {path_prefix}.faiss  ({self.index.ntotal} vectors, dim={self.d})")

    # ─────────────────────────────────────────────────────────────────────────
    def stats(self) -> dict:
        """Return basic statistics about the index."""
        categories = {}
        for doc in self.metadata:
            cat = doc.get("category", "unknown")
            categories[cat] = categories.get(cat, 0) + 1
        return {
            "total_documents": len(self.metadata),
            "embedding_dim"  : self.d,
            "by_category"    : categories,
        }


# ═══════════════════════════════════════════════════════════════════════════════
# Demo corpus
# ═══════════════════════════════════════════════════════════════════════════════

CORPUS = [
    # AI / Embeddings
    {"text": "FAISS is a library for efficient similarity search of dense vectors.", "category": "ai",          "difficulty": "intermediate"},
    {"text": "Sentence Transformers encode sentences into fixed-size dense vectors.", "category": "ai",          "difficulty": "beginner"},
    {"text": "Cosine similarity measures the angle between two embedding vectors.",   "category": "ai",          "difficulty": "beginner"},
    {"text": "Vector databases like Pinecone and ChromaDB store embeddings at scale.","category": "ai",          "difficulty": "intermediate"},
    {"text": "RAG pipelines retrieve relevant documents before generating answers.",   "category": "ai",          "difficulty": "advanced"},
    {"text": "BERT generates contextualised word embeddings using transformers.",      "category": "ai",          "difficulty": "intermediate"},
    # Programming
    {"text": "Python list comprehensions offer a concise way to create lists.",       "category": "programming", "difficulty": "beginner"},
    {"text": "Async/await in Python enables non-blocking IO operations.",             "category": "programming", "difficulty": "intermediate"},
    {"text": "Docker containers package applications with all dependencies.",          "category": "programming", "difficulty": "intermediate"},
    {"text": "REST APIs use HTTP methods: GET, POST, PUT, DELETE.",                   "category": "programming", "difficulty": "beginner"},
    # Science
    {"text": "Photosynthesis converts sunlight and CO₂ into glucose and oxygen.",     "category": "science",     "difficulty": "beginner"},
    {"text": "Quantum entanglement links particles so their states are correlated.",   "category": "science",     "difficulty": "advanced"},
    {"text": "DNA uses four bases: Adenine, Thymine, Guanine, and Cytosine.",         "category": "science",     "difficulty": "beginner"},
    {"text": "The Standard Model describes fundamental particles and forces.",         "category": "science",     "difficulty": "advanced"},
    # History
    {"text": "The French Revolution began in 1789 with the storming of the Bastille.","category": "history",    "difficulty": "beginner"},
    {"text": "The Industrial Revolution transformed manufacturing in 18th-century Britain.", "category": "history", "difficulty": "intermediate"},
    {"text": "World War II ended in 1945 with Allied victory in Europe and the Pacific.", "category": "history", "difficulty": "beginner"},
]


# ═══════════════════════════════════════════════════════════════════════════════
# Main demo
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("Day 4 – Script 05: Full Semantic Search Engine")
    print("=" * 60)

    # ── Build engine ──────────────────────────────────────────────────────────
    engine = SemanticSearchEngine("all-MiniLM-L6-v2")
    engine.build_index(CORPUS)

    # ── Stats ─────────────────────────────────────────────────────────────────
    print("\n── Engine Statistics ──")
    stats = engine.stats()
    print(f"  Total documents : {stats['total_documents']}")
    print(f"  Embedding dim   : {stats['embedding_dim']}")
    print(f"  By category     : {stats['by_category']}")

    # ── Searches ──────────────────────────────────────────────────────────────
    def demo_search(query, k=3, filters=None):
        filter_str = f"  filters={filters}" if filters else ""
        print(f"\n❓ '{query}'{filter_str}")
        results = engine.search(query, k=k, filters=filters)
        if not results:
            print("  (no results)")
        for r in results:
            meta = f"[{r['category']} | {r['difficulty']}]"
            print(f"  [{r['score']:.4f}] {meta:30s} {r['text'][:65]}")

    print("\n──────────────────────────────────────────────────────────")
    print("Search demonstrations")
    print("──────────────────────────────────────────────────────────")

    demo_search("how does vector similarity search work?")
    demo_search("how does vector similarity search work?", filters={"category": "ai"})
    demo_search("building APIs and backend services", filters={"category": "programming"})
    demo_search("beginner-friendly science topics", filters={"difficulty": "beginner"})
    demo_search("advanced machine learning techniques", filters={"category": "ai", "difficulty": "advanced"})
    demo_search("major historical events in the 20th century", k=4)

    # ── Incremental add ───────────────────────────────────────────────────────
    print("\n── Incremental Document Addition ──")
    new_docs = [
        {"text": "Kubernetes orchestrates containerised applications at scale.", "category": "programming", "difficulty": "advanced"},
        {"text": "Climate change is accelerating due to greenhouse gas emissions.", "category": "science", "difficulty": "beginner"},
    ]
    engine.add_documents(new_docs)
    demo_search("container orchestration systems")

    # ── Save & Reload ─────────────────────────────────────────────────────────
    print("\n── Save & Reload Engine ──")
    engine.save("semantic_engine")

    engine2 = SemanticSearchEngine.__new__(SemanticSearchEngine)
    engine2.model    = engine.model   # reuse loaded model
    engine2.index    = None
    engine2.metadata = []
    engine2.d        = None
    engine2.load("semantic_engine")

    print("\nVerification search after reload:")
    for r in engine2.search("efficient vector indexing", k=2):
        print(f"  [{r['score']:.4f}] {r['text']}")

    # ── Clean up demo files ───────────────────────────────────────────────────
    for f in ["semantic_engine.faiss", "semantic_engine_meta.json",
              "corpus_index.faiss", "corpus_meta.json", "demo_index.faiss"]:
        if os.path.exists(f):
            os.remove(f)

    # ── Interactive REPL ──────────────────────────────────────────────────────
    print("""
──────────────────────────────────────────────────────────
Interactive Search REPL
  Type a query to search all documents.
  Prefix with @<category> to filter: @ai what are embeddings?
  Type 'exit' or 'quit' to stop.
──────────────────────────────────────────────────────────""")

    while True:
        try:
            raw = input("\nSearch> ").strip()
        except (EOFError, KeyboardInterrupt):
            break

        if not raw or raw.lower() in ("exit", "quit", "q"):
            break

        # Parse optional category filter
        filters = None
        query   = raw
        if raw.startswith("@"):
            parts = raw.split(" ", 1)
            cat   = parts[0][1:]
            query = parts[1] if len(parts) > 1 else ""
            if query:
                filters = {"category": cat}
            else:
                print(f"  (please enter a query after @{cat})")
                continue

        if not query:
            continue

        results = engine.search(query, k=5, filters=filters)
        if not results:
            print("  No results found.")
        for r in results:
            meta = f"[{r['category']} | {r['difficulty']}]"
            print(f"  [{r['score']:.4f}] {meta:30s} {r['text']}")

    print("\n✅  Semantic search engine demo complete.")

---
## 6. Vector Database Comparison: FAISS vs ChromaDB vs Pinecone

| Feature | **FAISS** | **ChromaDB** | **Pinecone** |
|---|---|---|---|
| Type | Library | Embedded / Server DB | Managed Cloud |
| Cost | Free (open-source) | Free (open-source) | Free tier + paid plans |
| Metadata filtering | Manual (parallel dict) | Built-in | Built-in |
| Persistence | Manual (`write_index`) | Automatic | Automatic |
| Scalability | Single machine | Single machine/server | Multi-billion vectors |
| Setup complexity | Low | Very Low | Low (API key) |
| Best for | Research, custom pipelines | Local dev, prototypes | Production at scale |

### Decision Guide
```
Need a quick prototype or research?     → FAISS
Need metadata filters + local storage?  → ChromaDB
Need production scale (>10M vectors)?   → Pinecone
Need full control + no cloud?           → FAISS or ChromaDB
```


### ChromaDB Quick Demo (Optional)
Install: `pip install chromadb`


In [ ]:
# Optional ChromaDB demo — install first:
# !pip install chromadb -q

try:
    import chromadb
    from chromadb.utils import embedding_functions

    chroma_client = chromadb.Client()  # In-memory client

    ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

    collection = chroma_client.create_collection(
        name="day4_docs",
        embedding_function=ef,
    )

    collection.add(
        documents=[
            "Python is widely used in data science.",
            "The Colosseum is in Rome, Italy.",
            "PyTorch enables GPU-accelerated deep learning.",
            "FAISS is great for fast vector search.",
        ],
        metadatas=[
            {"category": "programming"},
            {"category": "travel"},
            {"category": "ml"},
            {"category": "ml"},
        ],
        ids=["doc1", "doc2", "doc3", "doc4"],
    )

    # Query with BUILT-IN metadata filter (no manual post-filtering needed)
    results = collection.query(
        query_texts=["machine learning frameworks"],
        n_results=2,
        where={"category": "ml"},
    )
    print("ChromaDB results (category=ml):")
    for doc in results["documents"][0]:
        print(f"  {doc}")

except ImportError:
    print("chromadb not installed. Run: pip install chromadb")


---
## Summary

| Concept | Key Takeaway |
|---|---|
| **Embeddings** | Dense vectors encoding text meaning; similar texts have similar vectors |
| **Sentence Transformers** | Free library; `model.encode()` is all you need; no GPU required |
| **Cosine Similarity** | Angle between vectors; range −1 to +1; scores >0.7 = semantically similar |
| **FAISS** | Ultra-fast similarity search; `IndexFlatIP` for exact, `IndexIVFFlat` for ANN |
| **Metadata Filtering** | FAISS = parallel Python list; ChromaDB/Pinecone = built-in |
| **Semantic Search** | Query → embed → FAISS search → rank by cosine score → return top-k |

## Further Reading
- [Sentence Transformers Docs](https://www.sbert.net/)
- [FAISS GitHub Repository](https://github.com/facebookresearch/faiss)
- [FAISS Index Types Wiki](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)
- [ChromaDB Docs](https://docs.trychroma.com/)
- [Pinecone Docs](https://docs.pinecone.io/)
- [MTEB Leaderboard – Best Embedding Models](https://huggingface.co/spaces/mteb/leaderboard)
